In [2]:
from app.services.pipeline import coletar_dados

from time import perf_counter

username = "felipe.cruz"
password = "#Gladoscruz.9851"
analise = "compra por necessidade"

dfs = coletar_dados(username, password, analise)

Extractor criado com sucesso!
ordens_LE FABRI02 concluída em 11.41s (1 / 102)
ordens_CAPAS LE concluída em 17.59s (2 / 102)
apoio_compras_CST concluída em 25.01s (3 / 102)
apoio_compras_RV concluída em 27.49s (4 / 102)
ordens_ESTOF UL concluída em 28.24s (5 / 102)
apoio_compras_VSN concluída em 30.47s (6 / 102)
consCAPAS LE concluída em 34.41s (7 / 102)
apoio_compras_CSD concluída em 18.37s (8 / 102)
apoio_compras_VRG concluída em 25.10s (9 / 102)
consPLANEJADOS concluída em 41.07s (10 / 102)
ordens_PLANEJADOS concluída em 6.37s (11 / 102)
ordens_ACESSORIOS concluída em 5.90s (12 / 102)
consMETALURGIA concluída em 17.32s (13 / 102)
ordens_UL ACABA01 concluída em 8.83s (14 / 102)
apoio_compras_NEC concluída em 19.68s (15 / 102)
ordens_CC ACABA01 concluída em 17.17s (16 / 102)
apoio_compras_FNC concluída em 47.52s (17 / 102)
ordens_KIT concluída em 49.18s (18 / 102)
estoque concluída em 10.74s (19 / 102)
consCC FABRI01 concluída em 23.37s (20 / 102)
consACESSORIOS concluída em 11.00s (21

In [3]:
import pickle

with open("snapshot_dfs.pkl", "wb") as f:
    pickle.dump(dfs, f)

print("Snapshot salvo.")



Snapshot salvo.


In [2]:
import pickle

dfs = {}
with open("snapshot_dfs.pkl", "rb") as f:
    dfs = pickle.load(f)

In [ ]:
import pandas as pd
import networkx as nx

print(f"{dfs.keys()}\n")


def sanitizar_dataframe(df, limite=0.8):
    df = df.copy()

    for col in df.columns:
        serie = df[col].astype(str).str.strip()

        tentativa_data = pd.to_datetime(
            serie, errors="coerce", dayfirst=True, format="%d/%m/%Y"
        )
        if tentativa_data.notna().mean() > limite:
            df[col] = tentativa_data
            continue

        serie_num = serie.str.replace(".", "", regex=False).str.replace(
            ",", ".", regex=False
        )
        tentativa_num = pd.to_numeric(serie_num, errors="coerce")
        if tentativa_num.notna().mean() > limite:
            df[col] = tentativa_num
            continue

        df[col] = serie.replace({"": None})

    return df


def calc_data(dfs):

    ## Ajuste Ordens ##
    ordens = sanitizar_dataframe(dfs.get("ordens"))
    ordens = ordens[
        [
            "Cliente",
            "Fábrica",
            "Ordem",
            "Pedido",
            "Item",
            "Saldo",
            "Representante",
            "Entrega Pedido",
            "Data Abertura",
        ]
    ]
    ordens = ordens.rename(columns={"Ordem": "Ordem Prod", "Saldo": "Saldo Prod"})

    colunas = ["Entrega Pedido", "Data Abertura"]
    for col in colunas:
        # Remove o ponto e garante que a coluna seja tratada como string
        ordens[col] = (
            ordens[col]
            .astype(str)
            .str.strip()
            .str.replace(r"[^\d]", "", regex=True)  # remove tudo que não for número
            .pipe(lambda s: pd.to_datetime(s, format="%d%m%Y", errors="coerce"))
        )
    ## Ajuste Consumo ##
    consumo = sanitizar_dataframe(dfs.get("cons"))

    consumo["Item"] = consumo["Item"].str.split("-").str[0].str.strip()
    print(consumo.columns)
    consumo = consumo[["Item", "Baixa", "Consumo", "Local Prod.", "OP", "Familia"]]
    consumo = consumo.rename(columns={"OP": "Ordem Cons"})

    ######## Item Pai ##########

    consumo["item_pai"] = consumo["Ordem Cons"].map(
        ordens.set_index("Ordem Prod")["Item"]
    )
    ######## Importa dados ##########
    consumo = consumo.merge(ordens[["Item", "Ordem Prod"]], on="Item", how="left")

    consumo = consumo.merge(
        ordens[
            [
                "Cliente",
                "Fábrica",
                "Ordem Prod",
                "Pedido",
                "Saldo Prod",
                "Representante",
                "Entrega Pedido",
                "Data Abertura",
            ]
        ].rename(columns={"Ordem Prod": "Ordem", "Saldo Prod": "Saldo"}),
        left_on="Ordem Cons",
        right_on="Ordem",
        how="left",
    )

    ######## Ordena as colunas ##########

    consumo = consumo[
        [
            "Ordem Prod",
            "Item",
            "Consumo",
            "Ordem Cons",
            "item_pai",
            "Saldo",
            "Pedido",
            "Representante",
            "Entrega Pedido",
            "Local Prod.",
            "Familia",
            "Cliente",
            "Fábrica",
            "Ordem",
            "Data Abertura",
            "Baixa",
        ]
    ]
    ######## Calculos Baseados em estoque ##########
    estoque = sanitizar_dataframe(dfs.get("estoque"))
    consumo["estoque"] = (
        consumo["Item"].map(estoque.groupby("Item")["Qtde."].sum()).fillna(0)
    )
    print (consumo[["Ordem Cons", "Ordem Prod", "item_pai"]].head(50))

    csv_path = "CSV/"
    consumo.to_excel(csv_path + "consumo.xlsx", index=False)


calc_data(dfs)

dict_keys(['ordens', 'apoio_compras', 'cons', 'estoque'])

Index(['Tipo', 'Grupo', 'Item', 'Local Estoque', 'Baixa', 'Situação',
       'Den. Item', 'Consumo', 'Local Prod.', 'OP', 'Familia', 'Família'],
      dtype='str')
    Ordem Cons  Ordem Prod        item_pai
0      1746892         NaN  2TAPCOWFPCU1Z1
1      1746892         NaN  2TAPCOWFPCU1Z1
2      1746893         NaN  2TAPCOWFPCU1Z2
3      1746893         NaN  2TAPCOWFPCU1Z2
4      1750594         NaN  9BCARES0004236
5      1701420         NaN  9BCARES0004264
6      1722637         NaN  9BCARES0004270
7      1723960         NaN  9BCARES0004281
8      1741483         NaN  9BCARES0004283
9      1726471         NaN  9BCARES0004285
10     1739906         NaN  9BCARES0004287
11     1744441         NaN  9BCARES0004289
12     1746372         NaN  9BCARES0004291
13     1762883         NaN  9BCARES0004293
14     1750595         NaN  9BCARES0004295
15     1753581         NaN  9BCARES0004299
16     1753582         NaN  9BCARES0004301
17 

KeyboardInterrupt: 